# 既存20 RunのSQL再分析

## 要点

不整合条件は平均総トークンが約11.6%少ない一方、合格IDあたり投入量は約37.8%多い。
閾値直接・後続依存13 IDに合格数差の62.5%が集中するが、因果寄与率ではない。
本文の考察と保存証拠は report.md と claims.md を参照。

## 文脈と方法

全20 Runを保持し、observed_tokensを本報告の確定総量として採用する。
旧raw点・旧usageフラグ・旧裁定は変更しない。推論単位はRun、対応単位は固定10ペア。
新規観測・モデル呼出し・再採点は一切行わない。


## データ

### 1. 同じフォルダのSQLiteを読み取り専用で開く

In [1]:
from pathlib import Path
import sqlite3, json, hashlib, numpy as np
from rebuild import query_map, rank
base=Path.cwd()
db=sqlite3.connect((base/'analysis.sqlite').as_uri()+'?mode=ro&immutable=1',uri=True)
db.row_factory=sqlite3.Row
queries=query_map()
def query(name): return [dict(r) for r in db.execute(queries[name])]
runs=query('runs'); pairs=query('pairs')
assert len(runs)==20 and len(pairs)==10
assert sum(r['total_tokens'] for r in runs)==134321987
print({'Run数':len(runs),'ペア数':len(pairs),'確定総トークン':sum(r['total_tokens'] for r in runs)})


{'Run数': 20, 'ペア数': 10, '確定総トークン': 134321987}


### 2. 元ケースから57 IDの得点を再構成する

In [2]:
scores={}
for row in db.execute("SELECT run_id,evaluation_id,MIN(status='pass') passed,COUNT(*) n FROM case_results GROUP BY run_id,evaluation_id"):
    scores.setdefault(row['run_id'],[]).append((row['evaluation_id'],row['passed'],row['n']))
assert sum(len(s) for s in scores.values())==1140
assert db.execute('SELECT COUNT(*) FROM case_results').fetchone()[0]==1160
for r in runs:
    assert len(scores[r['run_id']])==57
    assert sum(s[1] for s in scores[r['run_id']])==r['passed']
    assert next(s[2] for s in scores[r['run_id']] if s[0]=='T-006-05')==2
print({'再構成したRun×ID':1140,'元ケース':1160,'T00605の両ケース規則':'一致'})


{'再構成したRun×ID': 1140, '元ケース': 1160, 'T00605の両ケース規則': '一致'}


## 結果

### 3. 全20件の総量と適合効率

In [3]:
summary=[]
for condition in ('normal','anti'):
    rr=[r for r in runs if r['condition']==condition]
    summary.append({'condition':condition,'n':len(rr),'tokens':sum(r['total_tokens'] for r in rr),
                    'passed':sum(r['passed'] for r in rr)})
n,a=summary
relative_tokens=a['tokens']/n['tokens']-1
relative_cost=(a['tokens']/a['passed'])/(n['tokens']/n['passed'])-1
assert n['tokens']==71287304 and a['tokens']==63034683
assert n['passed']==402 and a['passed']==258
print(summary)
print({'トークン相対差':round(relative_tokens,6),'合格IDあたり投入相対差':round(relative_cost,6)})


[{'condition': 'normal', 'n': 10, 'tokens': 71287304, 'passed': 402}, {'condition': 'anti', 'n': 10, 'tokens': 63034683, 'passed': 258}]
{'トークン相対差': -0.115766, '合格IDあたり投入相対差': 0.377761}


### 4. 判断記録・固定分岐と、依存群の合格数

In [4]:
print(query('interpretations'))
groups=query('groups')
normal={r['dependency_group']:r['passed_ids'] for r in groups if r['condition']=='normal'}
anti={r['dependency_group']:r['passed_ids'] for r in groups if r['condition']=='anti'}
gaps={k:normal[k]-anti[k] for k in normal}
assert gaps=={'direct_threshold':30,'downstream_threshold':60,'other':54}
assert (gaps['direct_threshold']+gaps['downstream_threshold'])/sum(gaps.values())==0.625
print({'群別合格数差':gaps,'13IDへの集中割合':0.625})


[{'condition': 'anti', 'runs': 10, 'conflict_recorded': 10, 'recorded_threshold_yen': 500000, 'static_threshold_yen': 500000}, {'condition': 'normal', 'runs': 10, 'conflict_recorded': 0, 'recorded_threshold_yen': 1000000, 'static_threshold_yen': 1000000}]
{'群別合格数差': {'direct_threshold': 30, 'downstream_threshold': 60, 'other': 54}, '13IDへの集中割合': 0.625}


### 5. ペアを維持したbootstrapと1ペア除外

In [5]:
diff=np.array([[p['token_difference'],p['score_difference_pp']] for p in pairs])
indices=np.random.default_rng(20260910).integers(0,10,size=(20000,10))
interval=np.quantile(diff[indices].mean(axis=1),[.025,.975],axis=0)
loo=np.array([np.delete(diff,i,axis=0).mean(axis=0) for i in range(10)])
assert np.all(loo[:,1]<0)
print({'平均差':diff.mean(axis=0).tolist(),'95%区間_下上':interval.tolist(),
       '1ペア除外_得点差範囲':[float(loo[:,1].min()),float(loo[:,1].max())]})


{'平均差': [-825262.1, -25.263157894736842], '95%区間_下上': [[-1785974.5, -38.245614035087726], [319092.3, -13.508771929824565]], '1ペア除外_得点差範囲': [-28.265107212475634, -20.46783625730994]}


### 6. 条件内の相関と観測された反例

In [6]:
for condition in ('normal','anti'):
    rr=[r for r in runs if r['condition']==condition]
    xs=[r['total_tokens'] for r in rr];ys=[r['passed'] for r in rr]
    print(condition,{'Pearson':round(float(np.corrcoef(xs,ys)[0,1]),3),
                     'Spearman':round(float(np.corrcoef(rank(xs),rank(ys))[0,1]),3)})
print({'観測上の非劣位点':[r['planned_run'] for r in query('pareto')]})


normal {'Pearson': -0.47, 'Spearman': -0.587}
anti {'Pearson': 0.218, 'Spearman': 0.182}
{'観測上の非劣位点': ['anti-008', 'normal-007', 'normal-002']}


### 7. 応答回数と旧抽出・実行時期の感度

In [7]:
print(query('process'))
print(query('http_groups'))
for label,selected in [('前半',pairs[:5]),('後半',pairs[5:])]:
    print(label,{'n':len(selected),'平均トークン差':sum(r['token_difference'] for r in selected)/len(selected),
                '平均合格ID差':sum(r['passed_difference'] for r in selected)/len(selected)})


[{'condition': 'anti', 'runs': 10, 'total_tokens': 63034683, 'http_200': 1154, 'http_400': 4, 'tokens_per_200': 54622.775563258234, 'mean_elapsed_seconds': 736.5443646806996}, {'condition': 'normal', 'runs': 10, 'total_tokens': 71287304, 'http_200': 1270, 'http_400': 25, 'tokens_per_200': 56131.73543307087, 'mean_elapsed_seconds': 717.6602824333997}]
[{'condition': 'anti', 'has_http_400': 0, 'runs': 9, 'mean_tokens': 6005891.222222222, 'mean_passed': 25.0, 'http_400': 0}, {'condition': 'anti', 'has_http_400': 1, 'runs': 1, 'mean_tokens': 8981662.0, 'mean_passed': 33.0, 'http_400': 4}, {'condition': 'normal', 'has_http_400': 0, 'runs': 5, 'mean_tokens': 6455265.2, 'mean_passed': 42.2, 'http_400': 0}, {'condition': 'normal', 'has_http_400': 1, 'runs': 5, 'mean_tokens': 7802195.6, 'mean_passed': 38.2, 'http_400': 25}]
前半 {'n': 5, '平均トークン差': -20261.4, '平均合格ID差': -17.8}
後半 {'n': 5, '平均トークン差': -1630262.8, '平均合格ID差': -11.0}


### 8. コピーした元SQLiteとの原データ一致

In [8]:
original=sqlite3.connect((base/'source/analysis.sqlite').as_uri()+'?mode=ro&immutable=1',uri=True)
for table in ('runs','case_results','telemetry_refs','evaluations','provenance'):
    assert sorted(map(tuple,db.execute('SELECT * FROM '+table)))==sorted(original.execute('SELECT * FROM '+table))
print({'元5テーブル':'一致','元SQLite_SHA256':hashlib.sha256((base/'source/analysis.sqlite').read_bytes()).hexdigest()})
original.close();db.close()


{'元5テーブル': '一致', '元SQLite_SHA256': 'c5816c01c64f735e2cfb919df1d4fe8f1ce551d432c4201feff74fe9fe807d33'}


## 考察

- 矛盾の記録と詳細仕様優先が両立している。気づかなかったという説明とは区別する。
- 閾値依存群への失点集中は保存された操作経路と整合するが、因果寄与率ではない。
- 全件での消費量比較は、達成度・ばらつき・順序感度と併せて解釈する。
- 図と具体的trace事例、反証・限界を含む解釈は report.md にある。

![総トークンと項目充足率](figures/03-token-score.png)
